# Análise completa de um caso

### Carregamento do caso

In [1]:
import pandas as pd
import diplo_grafo.config as cfg
import re
from diplo_grafo.sentencing import get_sentences
from diplo_grafo.sentence import Sentence

# Ajuste para a pasta do projeto
base = cfg.SAMPLE_DATA_DIR

# Carrega os CSVs
cases = pd.read_csv(base / "cases.csv")
metadata = pd.read_csv(base / "metadata.csv")
data_dict = pd.read_csv(base / "data_dictionary.csv")

case = cases.iloc[0]

sentences = get_sentences(case)

print(case['case_text'])
print("\n==================================\n")
print("\n|| ".join([s.text for s in sentences]) + " ||\n")

A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparoscopic distal panc

2026-09-08 23:17:02.223 | INFO     | diplo_grafo.config:<module>:11 - PROJ_ROOT path is: /home/raphael/Desktop/faculdade/MC896_NLP/Diplotokens-P1/project 1


In [2]:
for s in sentences:
    s.tokenize()
    print(s.tokens)
    

['A', '44', 'year-old', 'woman', 'presented', 'with', 'a', '3', 'day', 'history', 'of', 'right', 'flank', 'and', 'lower', 'quadrant', 'abdominal', 'pain', 'associated', 'with', 'nausea', 'and', 'constipation']
['Her', 'past', 'medical', ',', 'family', 'and', 'medication', 'history', 'were', 'otherwise', 'non-contributory', 'and', 'her', 'physical', 'examination', 'was', 'unremarkable']
['She', 'underwent', 'contrast', 'enhanced', 'computed', 'tomography', ',', 'demonstrating', 'a', '6cm', 'cystic', 'lesion', 'between', 'the', 'stomach', 'and', 'body', 'tail', 'of', 'the', 'pancreas', '(', 'Fig', '1', ')']
['She', 'subsequently', 'underwent', 'EUS-FNA', ',', 'which', 'revealed', 'normal', 'pancreatic', 'echotexture', 'and', 'a', 'cyst', 'measuring', '6cm', 'x', '9cm', 'that', 'was', 'free', 'of', 'internal', 'septations', 'or', 'associated', 'masses', '(', 'Fig', '2', ')', 'but', 'compressed', 'the', 'stomach']
['FNA', 'of', 'the', 'cyst', 'demonstrated', 'no', 'evidence', 'of', 'malign

In [3]:
from diplo_grafo.clinical_terms import reconstruct_clinical_tokens

for sentence in sentences:
    sentence.tokenize()
    sentence.tokens = reconstruct_clinical_tokens(sentence.tokens)

for sentence in sentences:
    print(sentence.tokens)

['A', '44', 'year-old', 'woman', 'presented', 'with', 'a', '3', 'day', 'history', 'of', 'right flank', 'and', 'lower quadrant abdominal pain', 'associated', 'with', 'nausea', 'and', 'constipation']
['Her', 'past', 'medical', ',', 'family', 'and', 'medication', 'history', 'were', 'otherwise', 'non-contributory', 'and', 'her', 'physical', 'examination', 'was', 'unremarkable']
['She', 'underwent', 'contrast enhanced computed tomography', ',', 'demonstrating', 'a', '6cm', 'cystic lesion', 'between', 'the', 'stomach', 'and', 'body', 'tail', 'of', 'the', 'pancreas', '(', 'Fig', '1', ')']
['She', 'subsequently', 'underwent', 'EUS-FNA [endoscopic ultrasound-guided fine needle aspiration]', ',', 'which', 'revealed', 'normal', 'pancreatic echotexture', 'and', 'a', 'cyst', 'measuring', '6cm', 'x', '9cm', 'that', 'was', 'free', 'of', 'internal septations', 'or', 'associated masses', '(', 'Fig', '2', ')', 'but', 'compressed', 'the', 'stomach']
['FNA [fine needle aspiration]', 'of', 'the', 'cyst', '

### Construção do Grafo de Conhecimento (Nodes & Edges)

A partir das sentenças com tokens reconstruídos obtidas na célula anterior, estruturamos o Grafo de Conhecimento clínico seguindo o esquema do diagrama Mermaid em `docs/steps.md`, com entidades e relações em inglês (conforme o visualizador do grafo):

| Entidade no Mermaid (PT) | Tipo no Grafo (EN) | Prefixo ID |
|---|---|---|
| Paciente | `Patient` | `P` |
| Sintoma | `Symptom` | `S` |
| Doença | `Disease` | `D` |
| Exame | `Exam` | `E` |
| Resultado | `Result` | `R` |
| PartesCorpo | `Anatomy` | `A` |
| Medicação | `Medication` | `M` |
| Efeito | `Effect` | `Ef` |

#### Relações (Arestas):
* `Patient` -- `PRESENTS_WITH` --> `Symptom`
* `Patient` -- `UNDERWENT_EXAM` --> `Exam`
* `Patient` -- `DIAGNOSED_WITH` --> `Disease`
* `Patient` -- `HAS_PERSONAL_HISTORY` --> `Disease`
* `Patient` -- `HAS_FAMILY_HISTORY` --> `Disease`
* `Patient` -- `TAKES` --> `Medication`
* `Exam` -- `REVEALS` --> `Result`
* `Medication` -- `PRODUCES_EFFECT` --> `Effect`
* `Symptom` / `Disease` / `Result` -- `LOCATED_IN` --> `Anatomy`


In [4]:
# ==============================================================================
# LÓGICA DE CONSTRUÇÃO DO GRAFO (NODES & EDGES)
# ==============================================================================

# 1. Categorização semântica dos termos médicos
ENTITY_CATEGORIES = {
    "Symptom": {
        "lower quadrant abdominal pain", "abdominal pain", "nausea", "constipation",
        "pain", "dyspnea", "fever", "chest pain", "shortness of breath", "headache"
    },
    "Disease": {
        "mucinous pancreatic cystic neoplasm", "mucinous pancreatic cyst",
        "gastric duplication cyst", "GDC [gastric duplication cyst]",
        "systemic lupus erythematosus", "ventricular septal defect",
        "malignancy", "adenocarcinoma", "heart failure"
    },
    "Exam": {
        "contrast enhanced computed tomography", "computed tomography",
        "EUS-FNA [endoscopic ultrasound-guided fine needle aspiration]",
        "FNA [fine needle aspiration]", "Intraoperative endoscopy",
        "oesophagogastroduodenoscopy", "Final pathology", "physical examination",
        "biopsy", "ultrasound"
    },
    "Result": {
        "cystic lesion", "pancreatic echotexture", "cyst",
        "extracellular mucin", "carcinoembryonic antigen",
        "carbohydrate antigen", "internal septations", "associated masses"
    },
    "Anatomy": {
        "right flank", "stomach", "pancreas", "posterior wall",
        "lesser sac", "coeliac axis", "coeliac vessels", "abdomen"
    },
    "Medication": {
        "antibiotics", "chemotherapy"
    }
}

TERM_TO_TYPE = {
    term.lower(): cat
    for cat, terms in ENTITY_CATEGORIES.items()
    for term in terms
}

def extract_knowledge_graph(case_row, sentence_list):
    """
    Constrói os DataFrames de nós e arestas a partir das sentenças com tokens processados.
    """
    nodes = []
    edges = []
    node_ids = {}

    def get_or_create_node(label, node_type, attributes=""):
        key = (label.lower(), node_type)
        if key in node_ids:
            return node_ids[key]
        prefix = {
            "Patient": "P",
            "Symptom": "S",
            "Disease": "D",
            "Exam": "E",
            "Result": "R",
            "Anatomy": "A",
            "Medication": "M",
            "Effect": "Ef"
        }.get(node_type, "N")
        nid = f"{prefix}{len([k for k in node_ids if k[1] == node_type]) + 1}"
        node_ids[key] = nid
        nodes.append({
            "node_id": nid,
            "type": node_type,
            "label": label,
            "attributes": attributes
        })
        return nid

    def add_edge(source_id, target_id, relation, attributes=""):
        eid = f"e{len(edges) + 1}"
        edges.append({
            "edge_id": eid,
            "source_id": source_id,
            "target_id": target_id,
            "relation": relation,
            "attributes": attributes
        })

    # 1. Nó do Paciente
    patient_id = get_or_create_node(
        f"Patient {case_row['case_id']}",
        "Patient",
        f"age={case_row['age']}; gender={case_row['gender']}"
    )

    # 2. Extração sentencial
    for s in sentence_list:
        txt = s.text.lower()
        
        # Ignora frases não contributórias
        if "non-contributory" in txt or "were normal" in txt:
            continue

        found_entities = []
        for tok in s.tokens:
            tok_lower = tok.lower()
            if tok_lower in TERM_TO_TYPE:
                found_entities.append((tok, TERM_TO_TYPE[tok_lower]))

        # Extração de atributos escalares locais (tamanho, medições, duração)
        sizes = [t for t in s.tokens if ("cm" in t or "mm" in t) and any(c.isdigit() for c in t)]
        labs = [t for t in s.tokens if ("ng/ml" in t or "iu/ml" in t or "mg/dl" in t)]
        durations = [t for t in s.tokens if "day" in t and any(c.isdigit() for c in t)]

        symptoms = [e[0] for e in found_entities if e[1] == "Symptom"]
        diseases = [e[0] for e in found_entities if e[1] == "Disease"]
        exams = [e[0] for e in found_entities if e[1] == "Exam"]
        results = [e[0] for e in found_entities if e[1] == "Result"]
        anatomy = [e[0] for e in found_entities if e[1] == "Anatomy"]

        # Relação: Patient --PRESENTS_WITH--> Symptom
        if symptoms and any(w in txt for w in ["presented", "history", "pain", "complained", "had"]):
            sym_attr = f"duration={durations[0]}" if durations else ""
            for sym in symptoms:
                sid = get_or_create_node(sym, "Symptom", sym_attr)
                add_edge(patient_id, sid, "PRESENTS_WITH")
                # Symptom --LOCATED_IN--> Anatomy
                for anat in anatomy:
                    aid = get_or_create_node(anat, "Anatomy")
                    add_edge(sid, aid, "LOCATED_IN")

        # Relação: Patient --UNDERWENT_EXAM--> Exam
        if exams:
            for ex in exams:
                eid = get_or_create_node(ex, "Exam")
                add_edge(patient_id, eid, "UNDERWENT_EXAM")
                # Exam --REVEALS--> Result
                for res in results:
                    # Filtra achados negados sob "no evidence of"
                    if "no evidence of" in txt and res.lower() in txt.split("no evidence of")[1]:
                        continue
                    res_attr = ""
                    if sizes: res_attr += f"size={sizes[0]}; "
                    if labs: res_attr += f"value={labs[0]}; "
                    rid = get_or_create_node(res, "Result", res_attr.strip("; "))
                    add_edge(eid, rid, "REVEALS")
                    # Result --LOCATED_IN--> Anatomy
                    for anat in anatomy:
                        aid = get_or_create_node(anat, "Anatomy")
                        add_edge(rid, aid, "LOCATED_IN")

        # Relação: Patient --DIAGNOSED_WITH / HAS_HISTORY--> Disease
        if diseases:
            for dis in diseases:
                if "no evidence of" in txt and dis.lower() in txt.split("no evidence of")[1]:
                    continue
                did = get_or_create_node(dis, "Disease")
                if "history" in txt:
                    rel = "HAS_FAMILY_HISTORY" if "family" in txt else "HAS_PERSONAL_HISTORY"
                else:
                    rel = "DIAGNOSED_WITH"
                add_edge(patient_id, did, rel)
                # Disease --LOCATED_IN--> Anatomy
                for anat in anatomy:
                    aid = get_or_create_node(anat, "Anatomy")
                    add_edge(did, aid, "LOCATED_IN")

    return pd.DataFrame(nodes), pd.DataFrame(edges)

nodes_df, edges_df = extract_knowledge_graph(case, sentences)

# Salvar as tabelas para visualização
proc_dir = cfg.PROCESSED_DATA_DIR
proc_dir.mkdir(parents=True, exist_ok=True)
nodes_df.to_csv(proc_dir / "nodes.csv", index=False)
edges_df.to_csv(proc_dir / "edges.csv", index=False)

print(f"Grafo construído com sucesso!")
print(f"Total de Nós: {len(nodes_df)}")
print(f"Total de Arestas: {len(edges_df)}")
print(f"Arquivos exportados para: {proc_dir / 'nodes.csv'} e {proc_dir / 'edges.csv'}")


Grafo construído com sucesso!
Total de Nós: 21
Total de Arestas: 28
Arquivos exportados para: /home/raphael/Desktop/faculdade/MC896_NLP/Diplotokens-P1/project 1/data/processed/nodes.csv e /home/raphael/Desktop/faculdade/MC896_NLP/Diplotokens-P1/project 1/data/processed/edges.csv


In [5]:
print("=== TABELA DE NÓS (nodes.csv) ===")
display(nodes_df.head(15))

print("\n=== TABELA DE ARESTAS (edges.csv) ===")
display(edges_df.head(15))


=== TABELA DE NÓS (nodes.csv) ===
   node_id     type                                                          label               attributes
0       P1  Patient                                          Patient PMC5137649_01  age=44.0; gender=Female
1       S1  Symptom                                  lower quadrant abdominal pain                         
2       A1  Anatomy                                                    right flank                         
3       S2  Symptom                                                         nausea                         
4       S3  Symptom                                                   constipation                         
5       E1     Exam                          contrast enhanced computed tomography                         
6       R1   Result                                                  cystic lesion                 size=6cm
7       A2  Anatomy                                                        stomach                    

In [6]:
from IPython.display import display, Markdown

def generate_mermaid_code(nodes, edges):
    lines = ["```mermaid", "graph LR"]
    for _, r in nodes.iterrows():
        nid = r["node_id"]
        lbl = r["label"].replace('"', "'")
        ntype = r["type"]
        if ntype == "Patient":
            lines.append(f'    {nid}(("{lbl}"))')
        else:
            lines.append(f'    {nid}["{lbl} <br/> <i>({ntype})</i>"]')
    for _, r in edges.iterrows():
        src = r["source_id"]
        tgt = r["target_id"]
        rel = r["relation"]
        lines.append(f'    {src} -- "{rel}" --> {tgt}')
    lines.append("```")
    return "\n".join(lines)

mermaid_md = generate_mermaid_code(nodes_df, edges_df)
display(Markdown(mermaid_md))
print(mermaid_md)


<IPython.core.display.Markdown object>
```mermaid
graph LR
    P1(("Patient PMC5137649_01"))
    S1["lower quadrant abdominal pain <br/> <i>(Symptom)</i>"]
    A1["right flank <br/> <i>(Anatomy)</i>"]
    S2["nausea <br/> <i>(Symptom)</i>"]
    S3["constipation <br/> <i>(Symptom)</i>"]
    E1["contrast enhanced computed tomography <br/> <i>(Exam)</i>"]
    R1["cystic lesion <br/> <i>(Result)</i>"]
    A2["stomach <br/> <i>(Anatomy)</i>"]
    A3["pancreas <br/> <i>(Anatomy)</i>"]
    E2["EUS-FNA [endoscopic ultrasound-guided fine needle aspiration] <br/> <i>(Exam)</i>"]
    R2["pancreatic echotexture <br/> <i>(Result)</i>"]
    R3["cyst <br/> <i>(Result)</i>"]
    R4["internal septations <br/> <i>(Result)</i>"]
    R5["associated masses <br/> <i>(Result)</i>"]
    E3["FNA [fine needle aspiration] <br/> <i>(Exam)</i>"]
    E4["Intraoperative endoscopy <br/> <i>(Exam)</i>"]
    D1["gastric duplication cyst <br/> <i>(Disease)</i>"]
    A4["posterior wall <br/> <i>(Anatomy)</i>"]
    E5["oe